#   Feature Engineering



##  Imports

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

DATA = 'Datos/'  

---
##  Cargar CSVs exportados por los EDAs

In [5]:
history_pair    = pd.read_csv(DATA + 'history_per_pair.csv')

players         = pd.read_csv(DATA + 'players_clean.csv', parse_dates=['created'])
friends         = pd.read_csv(DATA + 'friends_clean.csv')
purchased_users = pd.read_csv(DATA + 'purchased_users.csv')
reviews_player  = pd.read_csv(DATA + 'reviews_per_player.csv')

games           = pd.read_csv(DATA + 'games_clean.csv', parse_dates=['release_date'])
ach_per_game    = pd.read_csv(DATA + 'achievements_per_game.csv')
prices_game     = pd.read_csv(DATA + 'prices_per_game.csv')
purchased_game  = pd.read_csv(DATA + 'purchased_per_game.csv')
reviews_game    = pd.read_csv(DATA + 'reviews_per_game.csv')

print('CSVs cargados:')
for name, df in [
    ('history_per_pair',   history_pair),
    ('players_clean',      players),
    ('friends_clean',      friends),
    ('purchased_users',    purchased_users),
    ('reviews_per_player', reviews_player),
    ('games_clean',        games),
    ('ach_per_game',       ach_per_game),
    ('prices_per_game',    prices_game),
    ('purchased_per_game', purchased_game),
    ('reviews_per_game',   reviews_game),
]:
    print(f'  {name:<22}: {df.shape[0]:>8,} filas x {df.shape[1]} cols')

CSVs cargados:
  history_per_pair      :  253,610 filas x 3 cols
  players_clean         :  377,014 filas x 5 cols
  friends_clean         :  424,683 filas x 6 cols
  purchased_users       :  102,548 filas x 5 cols
  reviews_per_player    :  196,698 filas x 3 cols
  games_clean           :   98,248 filas x 10 cols
  ach_per_game          :   50,537 filas x 6 cols
  prices_per_game       :   97,713 filas x 11 cols
  purchased_per_game    :   40,988 filas x 5 cols
  reviews_per_game      :   51,910 filas x 9 cols


---
##  Features derivadas que requieren dos fuentes

Estas son las únicas variables que se calculan aquí — no podían calcularse dentro de un solo EDA porque dependen de dos tablas distintas.

In [6]:
ach_clean = ach_per_game[
    (~ach_per_game['is_spam_game']) &
    (ach_per_game['outlier_cat'] == 'normal')
][['gameid', 'n_achievements']]

history_cr = history_pair.merge(ach_clean, on='gameid', how='inner')
history_cr['completion_rate'] = (
    history_cr['n_logros_par'] / history_cr['n_achievements']
).clip(0, 1).round(4)

avg_cr = (
    history_cr
    .groupby('gameid')['completion_rate']
    .agg(avg_completion_rate='mean', n_players_with_history='count')
    .reset_index()
)

print(f'Pares con completion_rate : {len(history_cr):,} (de {len(history_pair):,} totales)')
print(f'completion_rate median    : {history_cr["completion_rate"].median():.3f}')

F2P_MIN_OWNERS = 10
f2p = prices_game[['gameid','has_price']].merge(
    purchased_game[['gameid','n_owners_public']], on='gameid', how='outer'
)
f2p['has_price']       = f2p['has_price'].fillna(False)
f2p['n_owners_public'] = f2p['n_owners_public'].fillna(0)
f2p['is_f2p'] = (~f2p['has_price']) & (f2p['n_owners_public'] >= F2P_MIN_OWNERS)
print(f'is_f2p=True               : {f2p["is_f2p"].sum():,} juegos')

Pares con completion_rate : 168,552 (de 253,610 totales)
completion_rate median    : 0.222
is_f2p=True               : 6,712 juegos


---
##  Tabla de features de usuario

Una fila por `playerid`. Se usará en la tabla de regresion.

In [7]:
user_features = (
    players[['playerid','country','account_age_years','is_private']]
    .merge(friends[['playerid','n_friends','n_friends_log','social_tier']], on='playerid', how='left')
    .merge(purchased_users[['playerid','lib_size','user_type']], on='playerid', how='left')
    .merge(reviews_player[['playerid','n_reviews','is_hyperactive_player']], on='playerid', how='left')
)

defaults = {
    'n_friends': 0, 'n_friends_log': 0, 'social_tier': 'solitario',
    'lib_size': 0, 'user_type': 'casual',
    'n_reviews': 0, 'is_hyperactive_player': False
}
for col, val in defaults.items():
    if col in user_features.columns:
        user_features[col] = user_features[col].fillna(val)

print(f'user_features: {user_features.shape}')
miss = user_features.isnull().sum()
print('Faltantes:', miss[miss > 0].to_dict() if miss[miss > 0].any() else 'ninguno')
user_features.head(3)

user_features: (377014, 11)
Faltantes: ninguno


,playerid,country,account_age_years,is_private,n_friends,n_friends_log,social_tier,lib_size,user_type,n_reviews,is_hyperactive_player
0,76561198287452552,Brazil,10.03,False,0,0.000000,solitario,476.0,coleccionista,9.0,False
1,76561198040436563,Israel,14.93,False,316,5.758902,hub,836.0,coleccionista,31.0,True
2,76561198049686270,Unknown,14.46,False,718,6.577861,hub,0.0,casual,3.0,False


---
##  Tabla de features de juego

Una fila por `gameid`. Se usará en ambas tablas maestras.

In [8]:
game_features = (
    games
    .merge(ach_per_game[['gameid','n_achievements','is_spam_game','spam_ratio',
                          'outlier_cat','pct_generic']],
           on='gameid', how='left')
    .merge(prices_game[['gameid','usd_latest','usd_log','price_tier',
                         'has_price','no_eur_region']],
           on='gameid', how='left')
    .merge(purchased_game[['gameid','n_owners_public','n_owners_all','penetration_pct']],
           on='gameid', how='left')
    .merge(reviews_game[['gameid','n_reviews','n_reviews_log','avg_helpful',
                          'avg_engagement','pct_with_text','avg_review_len']],
           on='gameid', how='left')
    .merge(avg_cr, on='gameid', how='left')
    .merge(f2p[['gameid','is_f2p']], on='gameid', how='left')
)

defaults_g = {
    'price_tier': 'free_or_unknown', 'has_price': False, 'is_f2p': False,
    'n_owners_public': 0, 'is_spam_game': False, 'outlier_cat': 'normal',
    'primary_genre': 'Unknown'
}
for col, val in defaults_g.items():
    if col in game_features.columns:
        game_features[col] = game_features[col].fillna(val)

print(f'game_features: {game_features.shape}')
miss = game_features.isnull().sum()
print('Faltantes (top):')
print(miss[miss > 0].sort_values(ascending=False).head(10))
game_features.head(3)

game_features: (98248, 32)
Faltantes (top):
n_players_with_history    87153
avg_completion_rate       87153
n_owners_all              61077
penetration_pct           61077
pct_generic               47788
n_achievements            47788
spam_ratio                47788
avg_engagement            46499
n_reviews                 46499
pct_with_text             46499
dtype: int64


,gameid,title,release_date,release_year,primary_genre,genres_count,developers_count,publishers_count,supported_languages_count,is_playtest,...,penetration_pct,n_reviews,n_reviews_log,avg_helpful,avg_engagement,pct_with_text,avg_review_len,avg_completion_rate,n_players_with_history,is_f2p
0,3281560,Horror Game To Play With Friends! Playtest,2024-10-21,2024,Unknown,0,0,0,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,3280930,Eternals' Path Playtest,2024-10-17,2024,Unknown,0,0,0,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,3280770,ANGST: A TALE OF SURVIVAL - Singleplayer Playtest,2024-10-13,2024,Unknown,0,0,0,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


---
##  Tabla maestra — Regresion

**Target:** `n_logros_par` y `completion_rate`

In [9]:
game_cols_reg = ['gameid','primary_genre','release_year','n_achievements',
                 'is_spam_game','usd_log','price_tier','has_price','is_f2p',
                 'no_eur_region','n_reviews_log','avg_engagement','is_playtest']
game_cols_reg = [c for c in game_cols_reg if c in game_features.columns]

tabla_regression = (
    history_cr[['playerid','gameid','n_logros_par','completion_rate']]
    .merge(user_features, on='playerid', how='left')
    .merge(game_features[game_cols_reg], on='gameid', how='left')
)

print(f'tabla_regression: {tabla_regression.shape}')
print(f'Jugadores unicos: {tabla_regression["playerid"].nunique():,}')
print(f'Juegos unicos   : {tabla_regression["gameid"].nunique():,}')
print(f'Faltantes totales: {tabla_regression.isnull().sum().sum():,}')
print()
print('Target n_logros_par:')
print(tabla_regression['n_logros_par'].describe().round(2))
tabla_regression.head(3)

tabla_regression: (168552, 26)
Jugadores unicos: 4,596
Juegos unicos   : 11,138
Faltantes totales: 109,610

Target n_logros_par:
count    168552.00
mean         10.64
std          31.95
min           1.00
25%           3.00
50%           6.00
75%          15.00
max        4096.00
Name: n_logros_par, dtype: float64


,playerid,gameid,n_logros_par,completion_rate,country,account_age_years,is_private,n_friends,n_friends_log,social_tier,...,n_achievements,is_spam_game,usd_log,price_tier,has_price,is_f2p,no_eur_region,n_reviews_log,avg_engagement,is_playtest
0,76561197960272169,300,5,0.0926,Australia,22.51,False,1679.0,7.426549,hub,...,54.0,False,2.396986,mid,True,False,False,5.874931,8.107042,False
1,76561197960272169,620,7,0.1373,Australia,22.51,False,1679.0,7.426549,hub,...,51.0,False,2.396986,mid,True,False,False,8.043342,3.656491,False
2,76561197960272169,4000,15,0.5172,Australia,22.51,False,1679.0,7.426549,hub,...,29.0,False,2.396986,mid,True,False,False,8.963672,4.247152,False


---
## Tabla maestra — Clasificacion

**Target:** `is_popular` = top 25% por `n_owners_public`

In [10]:
tabla_clasificacion = game_features.copy()

threshold = tabla_clasificacion['n_owners_public'].quantile(0.75)
tabla_clasificacion['is_popular'] = tabla_clasificacion['n_owners_public'] >= threshold

print(f'tabla_clasificacion: {tabla_clasificacion.shape}')
print(f'Umbral is_popular (P75): {threshold:.0f} n_owners_public')
print(f'is_popular=True  : {tabla_clasificacion["is_popular"].sum():,}  ({tabla_clasificacion["is_popular"].mean()*100:.1f}%)')
print(f'Faltantes totales: {tabla_clasificacion.isnull().sum().sum():,}')
tabla_clasificacion.head(3)

tabla_clasificacion: (98248, 33)
Umbral is_popular (P75): 25 n_owners_public
is_popular=True  : 24,626  (25.1%)
Faltantes totales: 780,078


,gameid,title,release_date,release_year,primary_genre,genres_count,developers_count,publishers_count,supported_languages_count,is_playtest,...,n_reviews,n_reviews_log,avg_helpful,avg_engagement,pct_with_text,avg_review_len,avg_completion_rate,n_players_with_history,is_f2p,is_popular
0,3281560,Horror Game To Play With Friends! Playtest,2024-10-21,2024,Unknown,0,0,0,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1,3280930,Eternals' Path Playtest,2024-10-17,2024,Unknown,0,0,0,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2,3280770,ANGST: A TALE OF SURVIVAL - Singleplayer Playtest,2024-10-13,2024,Unknown,0,0,0,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False


---
##  Exportar

In [11]:
tabla_regression.to_csv(DATA + 'tabla_regression.csv', index=False)
tabla_clasificacion.to_csv(DATA + 'tabla_clasificacion.csv', index=False)

print('Exportadas:')
print(f'  tabla_regression.csv    : {tabla_regression.shape[0]:,} x {tabla_regression.shape[1]}')
print(f'  tabla_clasificacion.csv : {tabla_clasificacion.shape[0]:,} x {tabla_clasificacion.shape[1]}')
print()
print('Columnas tabla_regression:')
print(list(tabla_regression.columns))
print()
print('Columnas tabla_clasificacion:')
print(list(tabla_clasificacion.columns))

Exportadas:
  tabla_regression.csv    : 168,552 x 26
  tabla_clasificacion.csv : 98,248 x 33

Columnas tabla_regression:
['playerid', 'gameid', 'n_logros_par', 'completion_rate', 'country', 'account_age_years', 'is_private', 'n_friends', 'n_friends_log', 'social_tier', 'lib_size', 'user_type', 'n_reviews', 'is_hyperactive_player', 'primary_genre', 'release_year', 'n_achievements', 'is_spam_game', 'usd_log', 'price_tier', 'has_price', 'is_f2p', 'no_eur_region', 'n_reviews_log', 'avg_engagement', 'is_playtest']

Columnas tabla_clasificacion:
['gameid', 'title', 'release_date', 'release_year', 'primary_genre', 'genres_count', 'developers_count', 'publishers_count', 'supported_languages_count', 'is_playtest', 'n_achievements', 'is_spam_game', 'spam_ratio', 'outlier_cat', 'pct_generic', 'usd_latest', 'usd_log', 'price_tier', 'has_price', 'no_eur_region', 'n_owners_public', 'n_owners_all', 'penetration_pct', 'n_reviews', 'n_reviews_log', 'avg_helpful', 'avg_engagement', 'pct_with_text', 'avg

---
### Decisiones tomadas aqui

| Decision | Justificacion |
|---|---|
| `completion_rate` solo sobre juegos `outlier_cat='normal'` y `is_spam_game=False` | Los juegos spam distorsionan el ratio — ver `eda_achievements`. |
| `is_f2p` = sin precio + >= 10 owners publicos | Umbral conservador, ajustable con `F2P_MIN_OWNERS`. |
| Target `is_popular` = P75 de `n_owners_public` | Split 75/25. Ajustar si la distribucion completa lo justifica. |
| `n_owners_public` sobre `n_owners_all` | Los privados inflan n_owners 10-24% — ver `eda_purchased_games`. |